# Civil War News — Notebook 3: BERTopic Topic Modeling

Embeds the cleaned corpus, fits BERTopic (GPU-accelerated UMAP + HDBSCAN),
computes keyword-group cosine similarities, merges sentiment scores from
Notebook 2, and exports article-level and panel CSVs for R regressions.

**Prerequisites:** Run Notebook 1 first and place `news_proc.zip` in your
Google Drive folder (`DRIVE_BASE`). Notebook 2 is optional (set `SENTIMENT_CSV`
to its Drive path to merge sentiment scores).

**Storage:** Input (`news_proc.zip`) is read from Google Drive. All outputs
are saved to Colab ephemeral storage (`/content/data/`) and downloaded to
your PC — download them before the session ends.

**Outputs:** `bertopic_results_{model}.csv`, `bertopic_panel_{model}.csv`,
`topic_info_{model}.csv`, `embeddings_{model}.npy`, `bertopic_model_{model}/`.

**Contents:**
- Section 1: Install Packages & Check GPU
- Section 2: Configuration
- Section 3: Storage Setup
- Section 4: Load Data & Sample
- Section 5: Document Embeddings
- Section 6: UMAP
- Section 7: HDBSCAN
- Section 8: BERTopic Parameters
- Section 9: BERTopic Fit
- Section 10: Keyword Groups
- Section 11: Keyword-Group Cosine Similarity
- Section 12: Assemble Results & Export

## Section 1: Install Packages & Check GPU

RAPIDS cuML enables GPU-accelerated UMAP and HDBSCAN (3–5x faster than CPU).
Falls back to CPU automatically if RAPIDS is unavailable.


In [1]:
# --- RAPIDS (GPU UMAP / HDBSCAN) ---
%pip install --q \
    cuml-cu12 \
    cudf-cu12 --extra-index-url=https://pypi.nvidia.com

# --- Core NLP stack ---
%pip install --q \
    bertopic \
    sentence-transformers \
    datasets \
    huggingface_hub \
    symspellpy \
    openpyxl \
    pandas

print("GPU + core packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 19.6 MB/s eta 0:00:00
GPU + core packages installed.


In [2]:
import torch
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {VRAM_GB:.1f} GB')
    DEVICE = 0
else:
    print('No GPU detected. Go to Runtime > Change runtime type > T4 GPU')
    VRAM_GB = 0
    DEVICE  = -1

GPU:  NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, zipfile, hashlib, logging, time

# ── Scientific ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── ML / NLP ───────────────────────────────────────────────────────────────────
from datasets import load_from_disk
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import modules as ST_models
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from huggingface_hub import HfApi, login, hf_hub_download

# ── HF upload helper ───────────────────────────────────────────────────────────
def _hf_upload(local_path, repo_id, subfolder=None, repo_type='dataset'):
    """Upload file to HF; skip if already present. Pass subfolder=MODEL_NAME for model-specific files."""
    _api = HfApi()
    _api.create_repo(repo_id, private=True, repo_type=repo_type, exist_ok=True)
    _fname        = os.path.basename(local_path)
    _path_in_repo = f'{subfolder}/{_fname}' if subfolder else _fname
    try:
        _existing = list(_api.list_repo_files(repo_id, repo_type=repo_type))
    except Exception:
        _existing = []
    if _path_in_repo in _existing:
        print(f'[already on HF] {_path_in_repo}')
        return
    _sz = os.path.getsize(local_path) / 1e9
    print(f'Uploading {_path_in_repo} to {repo_id} ({_sz:.2f} GB) ...')
    _api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=_path_in_repo,
        repo_id=repo_id,
        repo_type=repo_type,
    )
    print(f'Saved: hf://datasets/{repo_id}/{_path_in_repo}')

print('Libraries loaded.')

In [ ]:
# Prevent Colab idle disconnect during long GPU runs (encoding + UMAP on 8.4M docs).
from IPython.display import Javascript
display(Javascript(
    'setInterval(() => { '
    'const btn = document.querySelector("colab-toolbar-button#connect"); '
    'if (btn) btn.click(); }, 60000)'
))
print('Keep-alive active (reconnects every 60 s).')

## Section 2: Configuration

Edit the parameters in this cell. Nothing below the divider needs changing.


In [ ]:
RANDOM_SEED = 42

# ======================================================================
# EMBEDDING MODEL
#   ── Historical (recommended for publication) ──────────────────────────
#   2  emanjavacas/MacBERTh                     768-dim  Historical English 1450-1950
#   3  Livingwithmachines/bert_1760_1900        768-dim  Historical British newspapers
#   8  Livingwithmachines/bert_1850_1875        768-dim  Civil War era (1850-1875) — best temporal match
#   ── General-purpose baselines ─────────────────────────────────────────
#   1  sentence-transformers/all-MiniLM-L6-v2  384-dim  Fast baseline (384-token limit)
#   7  sentence-transformers/all-mpnet-base-v2 768-dim  Stronger general baseline
#   ── Retrieval-optimised (strong clustering performance) ───────────────
#   4  intfloat/e5-large-v2                    1024-dim  SOTA modern (uses 'passage:' prefix)
#   6  BAAI/bge-large-en-v1.5                  1024-dim  Strong for clustering tasks
#   ── Long-context (best for full newspaper articles) ───────────────────
#   5  nomic-ai/nomic-embed-text-v1.5           768-dim  8192-token window; best for long docs
EMBEDDING_MODEL = 8

# ======================================================================
# BERTOPIC
SEED_TOPICS_ON = False    # Bias model toward research-relevant clusters
N_TOPICS       = None     # None = auto; set integer to merge to fixed count
SAMPLE_FRAC    = 1        # Fraction of corpus (0.01 = test run ~84k docs; 1.0 = full ~8.4M)
MIN_CLUSTER_SIZE_OVERRIDE = 100  # None = auto; set int to force

# ======================================================================
# STORAGE
DRIVE_BASE    = '/content/drive/MyDrive/CivilWarNews'
LOCAL_BASE    = '/content/data'
HF_REPO       = 'patrickjcrawford/civil-war-news'
    # embeddings, UMAP, HDBSCAN arrays, BERTopic model, results CSVs

# Sentiment CSV from Notebook 2 (optional). Leave '' to skip.
SENTIMENT_CSV = ''

# ======================================================================
# (No edits needed below this line)
import os

MODEL_CONFIGS = {
    2: {'model_id': 'emanjavacas/MacBERTh',
        'name': 'macberth',       'dim': 768,  'pca_components': 64,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': True,  'trust_remote_code': False,
        'encode_batch_size': 2048},
    3: {'model_id': 'Livingwithmachines/bert_1760_1900',
        'name': 'bert_newspapers','dim': 768,  'pca_components': 64,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': True,  'trust_remote_code': False,
        'encode_batch_size': 2048},
    8: {'model_id': 'Livingwithmachines/bert_1850_1875',
        'name': 'bert_1850_1875','dim': 768,  'pca_components': 64,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': True,  'trust_remote_code': False,
        'encode_batch_size': 2048},
    1: {'model_id': 'sentence-transformers/all-MiniLM-L6-v2',
        'name': 'minilm',         'dim': 384,  'pca_components': 32,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': False, 'trust_remote_code': False,
        'encode_batch_size': 4096},
    7: {'model_id': 'sentence-transformers/all-mpnet-base-v2',
        'name': 'mpnet',          'dim': 768,  'pca_components': 32,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': False, 'trust_remote_code': False,
        'encode_batch_size': 2048},
    4: {'model_id': 'intfloat/e5-large-v2',
        'name': 'e5_large',       'dim': 1024, 'pca_components': 64,
        'doc_prefix': 'passage: ','kw_prefix': 'query: ',
        'needs_manual_pooling': False, 'trust_remote_code': False,
        'encode_batch_size': 1024},
    6: {'model_id': 'BAAI/bge-large-en-v1.5',
        'name': 'bge_large',      'dim': 1024, 'pca_components': 64,
        'doc_prefix': None,       'kw_prefix': None,
        'needs_manual_pooling': False, 'trust_remote_code': False,
        'encode_batch_size': 1024},
    5: {'model_id': 'nomic-ai/nomic-embed-text-v1.5',
        'name': 'nomic',          'dim': 768,  'pca_components': 32,
        'doc_prefix': 'search_document: ', 'kw_prefix': 'search_query: ',
        'needs_manual_pooling': False, 'trust_remote_code': True,
        'encode_batch_size': 256},
}

CFG            = MODEL_CONFIGS[EMBEDDING_MODEL]
MODEL_NAME     = CFG['name']
BASE_PATH      = LOCAL_BASE

NEWS_PROC_PATH  = f'{LOCAL_BASE}/news_proc'
EMBEDDINGS_PATH = f'{BASE_PATH}/embeddings_{MODEL_NAME}.npy'
MODEL_PATH      = f'{BASE_PATH}/bertopic_model_{MODEL_NAME}'
RESULTS_CSV     = f'{BASE_PATH}/bertopic_results_{MODEL_NAME}.csv'
PANEL_CSV       = f'{BASE_PATH}/bertopic_panel_{MODEL_NAME}.csv'
TOPIC_INFO_CSV  = f'{BASE_PATH}/topic_info_{MODEL_NAME}.csv'

print(f'Embedding : {CFG["model_id"]} ({CFG["dim"]}-dim)')
print(f'Model name: {MODEL_NAME}')
print(f'Sample    : {SAMPLE_FRAC*100:.1f}%')
print(f'Input     : {DRIVE_BASE}/news_proc.zip  →  {NEWS_PROC_PATH}')
print(f'Outputs   : hf://datasets/{HF_REPO}/  +  hf://datasets/{HF_REPO}/')
print(f'Sentiment : {SENTIMENT_CSV if SENTIMENT_CSV else "(not merging)"}')

## Section 3: Storage Setup

Mounts Google Drive, reads `news_proc.zip` from `DRIVE_BASE`, and extracts
it to Colab ephemeral storage. All outputs are written to `/content/data/`
and downloaded to your PC in Section 12.

In [ ]:
def _is_hf_dataset(path):
    return (os.path.exists(f'{path}/dataset_info.json') or
            os.path.exists(f'{path}/dataset_dict.json'))

# ── Hugging Face login ─────────────────────────────────────────────────────
try:
    from google.colab import userdata as _ud
    login(token=_ud.get('HF_TOKEN'), add_to_git_credential=False)
    print('[OK] Logged in to Hugging Face.')
except Exception as _hf_e:
    print(f'HF login: {_hf_e}')
    print('Add HF_TOKEN to Colab Secrets (key icon in sidebar).')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.makedirs(LOCAL_BASE, exist_ok=True)
print(f'Drive mounted. Reading input from: {DRIVE_BASE}')

_drive_zip = f'{DRIVE_BASE}/news_proc.zip'

if not _is_hf_dataset(NEWS_PROC_PATH):
    if os.path.exists(_drive_zip):
        print(f'Extracting news_proc.zip → {LOCAL_BASE} ...')
        with zipfile.ZipFile(_drive_zip, 'r') as _zf:
            _zf.extractall(LOCAL_BASE)
        print('Extracted.')
    else:
        print(f'news_proc.zip not found on Drive. Trying HuggingFace ({HF_REPO}) ...')
        try:
            _hf_zip = hf_hub_download(repo_id=HF_REPO, filename='news_proc.zip',
                                      repo_type='dataset')
            print(f'Extracting from HF cache → {LOCAL_BASE} ...')
            with zipfile.ZipFile(_hf_zip, 'r') as _zf:
                _zf.extractall(LOCAL_BASE)
            print('Extracted.')
        except Exception as _e:
            raise FileNotFoundError(
                f'news_proc.zip not found on Drive ({_drive_zip}) or HF ({HF_REPO}). '
                f'Run Notebook 1 first.'
            ) from _e

if not _is_hf_dataset(NEWS_PROC_PATH):
    raise FileNotFoundError(f'Dataset not found after extraction: {NEWS_PROC_PATH}')
print('[OK] news_proc found')

if SENTIMENT_CSV:
    status = 'OK' if os.path.exists(SENTIMENT_CSV) else 'NOT FOUND'
    print(f'[{status}] sentiment CSV: {SENTIMENT_CSV}')
    if status == 'NOT FOUND':
        print('     Run Notebook 2 first, or set SENTIMENT_CSV to its Drive path.')
else:
    print('[--] sentiment merge skipped (SENTIMENT_CSV is empty)')

## Section 4: Load Data & Sample

In [ ]:
news_proc = load_from_disk(NEWS_PROC_PATH)
n_sample  = int(len(news_proc) * SAMPLE_FRAC)
news_sub  = news_proc.shuffle(seed=RANDOM_SEED).select(range(n_sample))
articles  = news_sub['article']
print(f'Loaded {len(news_proc):,} processed articles')
print(f'Corpus subset: {len(news_sub):,} ({SAMPLE_FRAC*100:.1f}%)')

### Checkpoint — restore all cached files from Drive after runtime disconnect

Place this immediately after Section 2 configuration. All path variables (DRIVE_BASE, LOCAL_BASE, MODEL_NAME) must be defined first. Each subsequent compute section checks for its file and skips if already present.

In [ ]:
import shutil, zipfile, os

_restore_files = [
    f'embeddings_{MODEL_NAME}.npy',
    f'embeddings_{MODEL_NAME}.key',
    f'umap_{MODEL_NAME}.npy',
]

for fname in _restore_files:
    dst = f'{LOCAL_BASE}/{fname}'
    if os.path.exists(dst):
        print(f'[already present] {fname}')
        continue
    try:
        src = hf_hub_download(repo_id=HF_REPO,
                              filename=f'{MODEL_NAME}/{fname}',
                              repo_type='dataset')
        shutil.copy2(src, dst)
        print(f'Restored from HF: {MODEL_NAME}/{fname}  ({os.path.getsize(dst)/1e9:.2f} GB)')
    except Exception:
        drive_src = f'{DRIVE_BASE}/{fname}'
        if os.path.exists(drive_src):
            print(f'Copying {fname} from Drive ...')
            shutil.copy2(drive_src, dst)
            print(f'  done  ({os.path.getsize(dst)/1e9:.2f} GB)')
        else:
            print(f'[not found] {fname} — will recompute')

# HDBSCAN labels
_hdbscan_npy   = f'{LOCAL_BASE}/hdbscan_labels_{MODEL_NAME}.npy'
_hdbscan_fname = f'hdbscan_labels_{MODEL_NAME}.npy'
if os.path.exists(_hdbscan_npy):
    print(f'[already present] {_hdbscan_fname}')
else:
    try:
        src = hf_hub_download(repo_id=HF_REPO,
                              filename=f'{MODEL_NAME}/{_hdbscan_fname}',
                              repo_type='dataset')
        shutil.copy2(src, _hdbscan_npy)
        print(f'Restored from HF: {MODEL_NAME}/{_hdbscan_fname}  ({os.path.getsize(_hdbscan_npy)/1e6:.1f} MB)')
    except Exception:
        _drive_zip = f'{DRIVE_BASE}/hdbscan_labels_{MODEL_NAME}.zip'
        _drive_npy = f'{DRIVE_BASE}/hdbscan_labels_{MODEL_NAME}.npy'
        if os.path.exists(_drive_zip):
            print(f'Copying + unzipping hdbscan_labels from Drive ...')
            shutil.copy2(_drive_zip, f'{LOCAL_BASE}/hdbscan_labels_{MODEL_NAME}.zip')
            with zipfile.ZipFile(f'{LOCAL_BASE}/hdbscan_labels_{MODEL_NAME}.zip', 'r') as zf:
                zf.extractall(LOCAL_BASE)
            os.remove(f'{LOCAL_BASE}/hdbscan_labels_{MODEL_NAME}.zip')
            print(f'  done  ({os.path.getsize(_hdbscan_npy)/1e6:.1f} MB)')
        elif os.path.exists(_drive_npy):
            shutil.copy2(_drive_npy, _hdbscan_npy)
            print(f'  done  ({os.path.getsize(_hdbscan_npy)/1e6:.1f} MB)')
        else:
            print(f'[not found] {_hdbscan_fname} — will recompute')

## Section 5: Document Embeddings

Encodes articles with the selected embedding model. Saves to disk and
reloads from cache on subsequent runs.

**Model notes:**
- **MacBERTh** (2): Historical English 1450–1950 — *recommended for publication*
- **bert_1760_1900** (3): Historical British newspapers — *best temporal match*
- **MiniLM** (1): 384-dim, fast, 384-token limit — use as speed baseline only
- **mpnet** (7): 768-dim, stronger modern baseline than MiniLM
- **e5-large** (4): 1024-dim, SOTA modern retrieval; adds `passage:` prefix
- **bge-large** (6): 1024-dim, strong clustering performance
- **nomic** (5): 768-dim, **8192-token window** — best choice if articles exceed 384 tokens

In [ ]:
import gc, time
from tqdm.auto import tqdm

def load_embedding_model(cfg):
    model_id = cfg['model_id']
    if cfg['needs_manual_pooling']:
        word_model    = ST_models.Transformer(model_id, max_seq_length=512)
        pooling_model = ST_models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True
        )
        model = SentenceTransformer(modules=[word_model, pooling_model], device='cuda')
    else:
        model = SentenceTransformer(
            model_id,
            device='cuda',
            trust_remote_code=cfg.get('trust_remote_code', False),
        )
    print(f'Loaded: {model_id} (dim={cfg["dim"]})')
    if DEVICE != -1:
        model = model.half()
    return model

def _effective_batch_size(cfg):
    base = cfg['encode_batch_size']
    if   VRAM_GB >= 60: return min(base * 2, 8192)
    elif VRAM_GB >= 30: return min(base * 2, 8192)
    elif VRAM_GB >= 14: return base
    return base

def encode_docs(texts, model, cfg, show_progress=True):
    if cfg['doc_prefix']:
        texts = [cfg['doc_prefix'] + t for t in texts]
    batch_size = _effective_batch_size(cfg)
    chunk_size = batch_size * 16
    n_docs     = len(texts)
    print(f'  docs={n_docs:,}  gpu_batch={batch_size}')

    all_embs, t0 = [], time.time()
    with tqdm(total=n_docs, unit='doc', disable=not show_progress) as pbar:
        for i in range(0, n_docs, chunk_size):
            chunk = texts[i : i + chunk_size]
            embs  = model.encode(chunk, batch_size=batch_size,
                                 show_progress_bar=False, convert_to_numpy=True,
                                 normalize_embeddings=True)
            all_embs.append(embs)
            torch.cuda.empty_cache(); gc.collect()
            pbar.update(len(chunk))
            done = i + len(chunk)
            rate = done / (time.time() - t0)
            eta  = (n_docs - done) / rate if rate > 0 else 0
            pbar.set_postfix(docs_s=f'{rate:,.0f}', eta=f'{eta:.0f}s')

    return np.concatenate(all_embs)

def encode_keywords(keywords, model, cfg):
    if cfg['kw_prefix']:
        keywords = [cfg['kw_prefix'] + k for k in keywords]
    return model.encode(keywords, convert_to_numpy=True, show_progress_bar=False,
                        normalize_embeddings=True)

embedding_model = load_embedding_model(CFG)

In [ ]:
def _embedding_cache_key(cfg, corpus_size, seed=42):
    raw = f"{cfg['model_id']}|{cfg.get('max_seq_length', 512)}|{corpus_size}|{seed}"
    return hashlib.md5(raw.encode()).hexdigest()

CACHE_KEY      = _embedding_cache_key(CFG, len(articles))
CACHE_KEY_PATH = EMBEDDINGS_PATH.replace('.npy', '.key')

def _cache_valid():
    if not os.path.exists(EMBEDDINGS_PATH) or not os.path.exists(CACHE_KEY_PATH):
        return False
    with open(CACHE_KEY_PATH) as f:
        return f.read().strip() == CACHE_KEY

if _cache_valid():
    print(f'Cache hit (key={CACHE_KEY[:8]}...): {EMBEDDINGS_PATH}')
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f'Embeddings shape: {embeddings.shape}')
elif os.path.exists(EMBEDDINGS_PATH):
    print(f'[WARNING] Key mismatch or missing — loading existing embeddings from disk.')
    print(f'  Delete {EMBEDDINGS_PATH} to force re-encoding if the model/corpus changed.')
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f'Embeddings shape: {embeddings.shape}')
else:
    print(f'Encoding {len(articles):,} articles with {CFG["model_id"]}...')
    embeddings = encode_docs(articles, embedding_model, CFG)
    np.save(EMBEDDINGS_PATH, embeddings)
    with open(CACHE_KEY_PATH, 'w') as f:
        f.write(CACHE_KEY)
    print(f'Saved: {EMBEDDINGS_PATH}  shape={embeddings.shape}')
    _hf_upload(EMBEDDINGS_PATH, HF_REPO, subfolder=MODEL_NAME)
    _hf_upload(CACHE_KEY_PATH, HF_REPO, subfolder=MODEL_NAME)

## Section 6: UMAP

**Pipeline:**
- **6a** RAPIDS imports (GPU/CPU fallback)
- **6b** UMAP model definition (PCA → UMAP)
- **6c** UMAP instance
- **6d** Pre-compute UMAP reduction (cached)

### Section 6a: RAPIDS Imports

In [ ]:
try:
    from cuml.manifold import UMAP as cuUMAP
    from cuml.cluster import HDBSCAN as cuHDBSCAN
    GPU_AVAILABLE = True
    print('RAPIDS cuML imports OK -- GPU acceleration enabled')
except Exception as e:
    GPU_AVAILABLE = False
    print(f'RAPIDS not available ({e}) -- using CPU fallback')

### Section 6b: UMAP Model Definition

Two-stage reduction: CPU PCA reduces embeddings to `pca_components`-dim,
then GPU (or CPU fallback) UMAP reduces to 5-dim. Downstream cells cache
both stages to disk so this is skipped on rerun.

In [ ]:
class UMAPWithPCA:
    # Two-stage reduction: PCA then UMAP, both preferring GPU (cuML) with
    # CPU fallback. cuML PCA is 5-10x faster than sklearn randomized SVD
    # on large matrices and keeps the GPU busy during what was previously
    # an idle CPU-only stage. X (fp32 embeddings) is freed after PCA to
    # reclaim ~13 GB before the UMAP run.

    # cuML-specific params that umap-learn does not accept.
    _CUML_ONLY_PARAMS = {'build_algo', 'hash_input', 'precomputed_knn', 'output_type'}

    def __init__(self, pca_n_components=32, umap_params=None, prefer_gpu=True):
        default_umap = dict(n_neighbors=15, n_components=5, min_dist=0.0,
                            metric='cosine', verbose=False)
        self.pca_n_components = pca_n_components
        self.umap_params      = {**default_umap, **(umap_params or {})}
        self.prefer_gpu       = prefer_gpu and GPU_AVAILABLE
        self.backend          = None
        self.pca_             = None
        self.umap_            = None
        self._is_fitted       = False
        self._cached_output   = None   # set externally to skip recomputation

    def fit_transform(self, X, y=None):
        if self._is_fitted and self._cached_output is not None:
            print(f'[UMAPWithPCA] Returning cached output {self._cached_output.shape}')
            return self._cached_output

        X = np.asarray(X).astype(np.float32)

        # -- Step 1: PCA (GPU preferred, CPU fallback) --------------------
        print(f'[UMAPWithPCA] PCA  {X.shape} -> {self.pca_n_components}-dim ...')
        _pca_on_gpu = False
        if self.prefer_gpu:
            try:
                import cupy as cp
                from cuml import PCA as cuPCA_
                X_gpu     = cp.asarray(X)
                self.pca_ = cuPCA_(n_components=self.pca_n_components)
                _X_pca    = self.pca_.fit_transform(X_gpu)
                X_pca     = _X_pca.get() if hasattr(_X_pca, 'get') else np.asarray(_X_pca)
                del X_gpu, _X_pca; cp.get_default_memory_pool().free_all_blocks()
                del X; gc.collect()
                _pca_on_gpu = True
                print(f'[UMAPWithPCA] GPU PCA done  shape={X_pca.shape}')
            except Exception as e:
                print(f'[UMAPWithPCA] GPU PCA failed ({e}) -- falling back to CPU')

        if not _pca_on_gpu:
            from sklearn.decomposition import PCA
            self.pca_ = PCA(n_components=self.pca_n_components, random_state=RANDOM_SEED)
            X_pca = self.pca_.fit_transform(X)
            del X; gc.collect()
            print(f'[UMAPWithPCA] CPU PCA done  shape={X_pca.shape}')

        # -- Step 2: UMAP (GPU preferred, CPU fallback) -------------------
        if self.prefer_gpu:
            try:
                import cupy as cp, cuml
                from cuml.manifold import UMAP as cuUMAP_
                # Flush PyTorch allocator so cuML RMM can use the freed memory.
                try:
                    import torch; torch.cuda.empty_cache()
                except Exception:
                    pass
                gc.collect()
                print('[UMAPWithPCA] GPU UMAP ...')
                X_gpu      = cp.asarray(X_pca)
                self.umap_ = cuUMAP_(**self.umap_params)
                with cuml.using_output_type('numpy'):
                    umap_emb = self.umap_.fit_transform(X_gpu)
                del X_gpu; cp.get_default_memory_pool().free_all_blocks()
                self.backend = 'gpu'
            except Exception as e:
                print(f'[UMAPWithPCA] GPU UMAP failed ({e}) -- falling back to CPU')
                self.prefer_gpu = False

        if not self.prefer_gpu:
            import umap as _umap
            cpu_params = {k: v for k, v in self.umap_params.items()
                          if k not in self._CUML_ONLY_PARAMS}
            print(f'[UMAPWithPCA] CPU UMAP  params={cpu_params}')
            self.umap_ = _umap.UMAP(**cpu_params)
            umap_emb   = self.umap_.fit_transform(X_pca)
            self.backend = 'cpu'

        self._is_fitted     = True
        self._cached_output = umap_emb
        print(f'[UMAPWithPCA] done  backend={self.backend}  output={umap_emb.shape}')
        return umap_emb

    def transform(self, X):
        if not self._is_fitted:
            raise RuntimeError('UMAPWithPCA not fitted -- call fit_transform() first.')
        X     = np.asarray(X).astype(np.float32)
        _raw  = self.pca_.transform(X)
        X_pca = _raw.get() if hasattr(_raw, 'get') else np.asarray(_raw)
        if self.backend == 'gpu':
            import cupy as cp, cuml
            X_gpu = cp.asarray(X_pca)
            with cuml.using_output_type('numpy'):
                emb = self.umap_.transform(X_gpu)
            del X_gpu; cp.get_default_memory_pool().free_all_blocks()
        else:
            emb = self.umap_.transform(X_pca)
        return emb

    def fit(self, X, y=None):
        self.fit_transform(X, y); return self

    def get_params(self, deep=True):
        return {'pca_n_components': self.pca_n_components,
                'umap_params': self.umap_params,
                'prefer_gpu': self.prefer_gpu}

print('UMAPWithPCA class defined')

### Section 6c: UMAP Instance

In [ ]:
umap_model = UMAPWithPCA(
    pca_n_components=CFG['pca_components'],
    umap_params={
        'n_neighbors':  15,
        'n_components': 5,
        'min_dist':     0.1,   # was 0.0
        'metric':       'cosine',
        'build_algo':   'nn_descent',
        'random_state': RANDOM_SEED,
        'verbose':      True,
    },
    prefer_gpu=True,
)
print('UMAP model defined')


### Section 6d: Pre-compute UMAP Reduction (Checkpoint)

Runs PCA + UMAP on the document embeddings and saves to disk. On rerun the
cached file is loaded and injected into `umap_model` so BERTopic skips
recomputation entirely.

In [ ]:
UMAP_PATH = f'{BASE_PATH}/umap_{MODEL_NAME}.npy'

if os.path.exists(UMAP_PATH):
    umap_embeddings = np.load(UMAP_PATH)
    umap_model._cached_output = umap_embeddings
    umap_model._is_fitted     = True
    print(f'UMAP cache loaded: {umap_embeddings.shape}  ({UMAP_PATH})')
else:
    import time as _t
    print(f'Running UMAP on {embeddings.shape[0]:,} x {embeddings.shape[1]} embeddings...')
    _t0 = _t.time()
    umap_embeddings = umap_model.fit_transform(embeddings)
    print(f'UMAP done in {_t.time()-_t0:.0f}s  shape={umap_embeddings.shape}')
    np.save(UMAP_PATH, umap_embeddings)
    print(f'Saved: {UMAP_PATH}')
    _hf_upload(UMAP_PATH, HF_REPO, subfolder=MODEL_NAME)

## Section 7: HDBSCAN

Define the HDBSCAN model. Then runs HDBSCAN on the cached UMAP output and saves cluster labels to disk.
On rerun cached labels are injected into `hdbscan_model` so BERTopic skips
clustering entirely. Clustering 8.4M docs takes ~26 min on an A100.

In [ ]:
class HDBSCANWithCache:
    """Wraps cuHDBSCAN or hdbscan.HDBSCAN and short-circuits to cached labels.
    BERTopic calls fit(X) then reads .labels_, so this is transparent to it."""

    def __init__(self, model):
        self._model         = model
        self._cached_labels = None
        self._is_fitted     = False
        self.labels_        = None
        self.probabilities_ = None

    def fit(self, X, y=None):
        if self._is_fitted and self._cached_labels is not None:
            print(f'[HDBSCANWithCache] Returning cached labels ({len(self._cached_labels):,} docs)')
            self.labels_ = self._cached_labels
            return self
        print(f'[HDBSCANWithCache] Fitting HDBSCAN on {X.shape} ...')
        self._model.fit(X)
        import cupy as cp
        _raw = self._model.labels_
        self.labels_        = _raw.get() if isinstance(_raw, cp.ndarray) else np.asarray(_raw)
        self._cached_labels = self.labels_.copy()
        self._is_fitted     = True
        return self

    def fit_predict(self, X, y=None):
        return self.fit(X, y).labels_

    def __getattr__(self, name):
        return getattr(self._model, name)


MIN_CLUSTER_SIZE = (MIN_CLUSTER_SIZE_OVERRIDE if MIN_CLUSTER_SIZE_OVERRIDE is not None
                    else max(50, int(len(articles) * 0.0001)))
print(f'min_cluster_size: {MIN_CLUSTER_SIZE}')

if GPU_AVAILABLE:
    from cuml.cluster import HDBSCAN as cuHDBSCAN
    _base_hdbscan = cuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=5,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=False,
        verbose=True,
    )
else:
    import hdbscan as _hdbscan_lib
    _base_hdbscan = _hdbscan_lib.HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE, min_samples=5,
        metric='euclidean', cluster_selection_method='eom',
        prediction_data=False,
    )

hdbscan_model = HDBSCANWithCache(_base_hdbscan)
print('HDBSCAN model defined (wrapped with cache)')

In [ ]:
HDBSCAN_PATH = f'{BASE_PATH}/hdbscan_labels_{MODEL_NAME}.npy'

if hdbscan_model._is_fitted and hdbscan_model._cached_labels is not None:
    _n_clusters = len(set(hdbscan_model._cached_labels.tolist())) - (
        1 if -1 in hdbscan_model._cached_labels else 0)
    print(f'HDBSCAN already loaded -- {_n_clusters} clusters, '
          f'{int((hdbscan_model._cached_labels == -1).sum()):,} noise docs. Skipping.')

elif os.path.exists(HDBSCAN_PATH):
    _labels = np.load(HDBSCAN_PATH)
    hdbscan_model._cached_labels = _labels
    hdbscan_model._is_fitted     = True
    hdbscan_model.labels_        = _labels
    _n_clusters = len(set(_labels.tolist())) - (1 if -1 in _labels else 0)
    print(f'HDBSCAN cache loaded: {_n_clusters} clusters, '
          f'{int((_labels == -1).sum()):,} noise docs  ({HDBSCAN_PATH})')

else:
    if 'umap_embeddings' not in dir():
        umap_embeddings = np.load(UMAP_PATH)
        print(f'Reloaded UMAP embeddings: {umap_embeddings.shape}')

    if hasattr(umap_model, 'umap_') and umap_model.umap_ is not None:
        del umap_model.umap_
        umap_model.umap_ = None
        if GPU_AVAILABLE:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        gc.collect()
        print('Freed cuML UMAP model from VRAM.')

    if GPU_AVAILABLE and 'embedding_model' in dir():
        _saved_emb_model = embedding_model
        del embedding_model
        torch.cuda.empty_cache()
        gc.collect()
        print('Freed embedding model from GPU before HDBSCAN.')

    if GPU_AVAILABLE:
        import cupy as cp
        _X = cp.asarray(umap_embeddings.astype(np.float32))
    else:
        _X = umap_embeddings.astype(np.float32)

    print(f'Running HDBSCAN on {_X.shape} ({"GPU cupy" if GPU_AVAILABLE else "CPU"})...')
    _t0 = time.time()
    hdbscan_model.fit(_X)
    del _X
    if GPU_AVAILABLE:
        cp.get_default_memory_pool().free_all_blocks()
    gc.collect()

    _labels     = hdbscan_model.labels_
    _n_clusters = len(set(_labels.tolist())) - (1 if -1 in _labels else 0)
    print(f'HDBSCAN done in {time.time()-_t0:.0f}s: '
          f'{_n_clusters} clusters, {int((_labels == -1).sum()):,} noise docs')
    np.save(HDBSCAN_PATH, _labels)
    print(f'Saved: {HDBSCAN_PATH}')
    _hf_upload(HDBSCAN_PATH, HF_REPO, subfolder=MODEL_NAME)

    if '_saved_emb_model' in dir():
        embedding_model = _saved_emb_model

## Section 8: BERTopic Parameters

- **8a** Vectorizer + c-TF-IDF
- **8b** Representation model

### Section 8a: Vectorizer + c-TF-IDF

BERTopic runs CountVectorizer on N_topics concatenated documents (one giant string per topic), not on 8.4M individual articles. So min_df is relative to the number of topics (~300-500): min_df=2 means a term must appear in at least 2 topic-documents. Bigrams are fine at this scale.

In [ ]:
vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_features=50_000,
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
print('Vectorizer and c-TF-IDF defined')

### Section 8b: Representation Model

Phase-1: no representation model. KeyBERT/MMR are applied in the update_topics cell below on a sample, after the full fit completes. Running KeyBERT inside fit_transform re-encodes representative docs while embeddings + DTM are all live — that alone can exhaust RAM.

In [ ]:
representation_model = None
print('Representation model: None (KeyBERT/MMR run post-fit on sample)')

## Section 9: BERTopic Fit

- **9a** Fit
- **9b** Phase-2 keyword refinement (optional)
- **9c** Save + download

### Section 9a: Fit

Fits BERTopic using cached UMAP and HDBSCAN outputs. A dummy embedding
array is passed so BERTopic skips re-encoding. Articles are truncated to
512 chars before fitting to cap the c-TF-IDF join spike at ~4 GB regardless
of original article length.

In [ ]:
logging.basicConfig(format='%(asctime)s %(levelname)s %(name)s  %(message)s',
                    datefmt='%H:%M:%S', level=logging.INFO, force=True)
for _log in ('bertopic', 'hdbscan', 'umap'):
    logging.getLogger(_log).setLevel(logging.DEBUG)

SEED_TOPICS = [
    ['slave', 'slavery', 'bondage', 'master', 'plantation', 'enslaved'],
    ['abolition', 'emancipation', 'freedman', 'free negro', 'manumission'],
    ['elect', 'vote', 'ballot', 'congress', 'candidate', 'partisan'],
    ['negro soldier', 'colored troop', 'contrabands', 'black regiment'],
    ['battle', 'army', 'regiment', 'commander', 'military campaign'],
] if SEED_TOPICS_ON else None

bertopic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    seed_topic_list=SEED_TOPICS,
    nr_topics=N_TOPICS,
    top_n_words=20,
    low_memory=True,
    calculate_probabilities=False,
    verbose=True,
)

# Free the large embeddings array and GPU model before fitting — both UMAP and HDBSCAN
# caches ignore their input, and CountVectorizer only reads article text.
_n_docs        = len(articles)
_emb_model_ref = embedding_model
del embedding_model, embeddings
torch.cuda.empty_cache()
gc.collect()
print(f'Freed embeddings (~{_n_docs * CFG["dim"] * 2 / 1e9:.1f} GB) and GPU model.')

# Truncate articles to the first 512 chars before fitting. Embeddings are already
# pre-computed from the full text so this only affects c-TF-IDF vocabulary.
# BERTopic's _extract_topics does ' '.join(all_articles_in_topic) per topic,
# creating a full second copy of all article text. Full newspaper articles can
# easily average 5,000+ chars, making that spike 80-170 GB. Truncating to 512
# chars caps the join at ~4 GB regardless of original article length. The
# BERT model embeddings were truncted the same.
_FIT_MAX_CHARS = 512
print(f'Truncating {_n_docs:,} articles to {_FIT_MAX_CHARS} chars for c-TF-IDF ...')
_articles_trunc = [a[:_FIT_MAX_CHARS] for a in articles]
del articles
gc.collect()
articles = _articles_trunc
del _articles_trunc
print(f'Truncation done. articles[0][:80] = {articles[0][:80]!r}')

# Tiny dummy: BERTopic skips re-encoding when embeddings are provided.
# UMAPWithPCA and HDBSCANWithCache return their cached output without reading it.
_dummy = np.zeros((_n_docs, 1), dtype=np.float16)

print(f'Fitting BERTopic on {_n_docs:,} articles (UMAP + HDBSCAN cached)...')
_t0 = time.time()
topics, probs = bertopic_model.fit_transform(articles, embeddings=_dummy)
del _dummy
gc.collect()
_elapsed = time.time() - _t0

n_topics = bertopic_model.get_topic_info().shape[0] - 1
print(f'\nBERTopic fit done in {_elapsed:.0f}s')
print(f'Topics found : {n_topics}')
print(f'Noise docs (-1): {sum(t == -1 for t in topics):,}')

embedding_model = _emb_model_ref  # restore for downstream cells

### Section 9b: Phase-2 Keyword Refinement (Optional)

Refines topic keywords and applies KeyBERT/MMR on a sample of the corpus.
Running on 500K docs instead of 8.4M keeps peak RAM under ~8 GB while still
producing high-quality bigram keywords. Set `UPDATE_TOPICS_SAMPLE = 0` to skip.

In [ ]:
UPDATE_TOPICS_SAMPLE = 0  # set to 0 to skip phase-2

if UPDATE_TOPICS_SAMPLE > 0:
    import random as _random

    _random.seed(RANDOM_SEED)

    # Sample indices, stratified by topic so small topics stay represented.
    _topic_arr = np.array(topics)
    _all_idx = list(range(len(articles)))
    _rng = _random.Random(RANDOM_SEED)
    _rng.shuffle(_all_idx)
    _sample_idx = _all_idx[:UPDATE_TOPICS_SAMPLE]
    _sample_idx.sort()

    _sample_docs = [articles[i] for i in _sample_idx]
    _sample_topics = [topics[i] for i in _sample_idx]

    # Free embeddings — not needed for update_topics (text + topic assignments only).
    # Embeddings freed in Section 9a; guard against NameError.
    if 'embeddings' in dir():
        del embeddings
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Phase-2: updating topics on {len(_sample_docs):,}-doc sample...")

    phase2_vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),  # bigrams restored for better keyword quality
        min_df=10,  # low enough to work on the sample
        max_df=0.85,
        max_features=50_000,
    )
    phase2_rep = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
    }

    _t1 = time.time()
    bertopic_model.update_topics(
        _sample_docs,
        topics=_sample_topics,
        vectorizer_model=phase2_vectorizer,
        representation_model=phase2_rep,
        top_n_words=20,
    )
    print(f"Phase-2 done in {time.time() - _t1:.0f}s")
    del _sample_docs, _sample_topics, _sample_idx
    gc.collect()

    topic_info = bertopic_model.get_topic_info()
    print(topic_info.head(10).to_string())
else:
    print("Phase-2 skipped (UPDATE_TOPICS_SAMPLE = 0).")
    topic_info = bertopic_model.get_topic_info()
    print(topic_info.head(10).to_string())

In [ ]:
# print('Reducing outliers (c-tf-idf strategy)...')
# new_topics = bertopic_model.reduce_outliers(articles, topics, strategy='c-tf-idf')
# bertopic_model.update_topics(articles, topics=new_topics)
# topics = new_topics
# print(f'Noise remaining: {sum(t == -1 for t in topics):,}')

### Section 9c: Save + Download

In [ ]:
topic_info = bertopic_model.get_topic_info()
topic_info.to_csv(TOPIC_INFO_CSV, index=False)
print(f'Topic info saved: {TOPIC_INFO_CSV}')
print(topic_info.head(10).to_string())

# BERTopic's safetensors path uses stdlib json which can't serialize numpy int64.
import json as _json
_orig_default = _json.JSONEncoder.default
def _numpy_default(self, obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    return _orig_default(self, obj)
_json.JSONEncoder.default = _numpy_default

bertopic_model.save(MODEL_PATH, serialization='safetensors',
                    save_ctfidf=True, save_embedding_model=embedding_model)

_json.JSONEncoder.default = _orig_default
print(f'Model saved: {MODEL_PATH}')

_topics_path = f'{BASE_PATH}/topics_{MODEL_NAME}.npy'
_probs_path  = f'{BASE_PATH}/probs_{MODEL_NAME}.npy'

np.save(_topics_path, np.array(topics))
if probs is not None and hasattr(probs, 'shape'):
    np.save(_probs_path, probs)
print('Arrays saved.')

# Upload all model-specific outputs under MODEL_NAME subfolder
_hf_upload(_topics_path,  HF_REPO, subfolder=MODEL_NAME)
if os.path.exists(_probs_path):
    _hf_upload(_probs_path, HF_REPO, subfolder=MODEL_NAME)
_hf_upload(TOPIC_INFO_CSV, HF_REPO, subfolder=MODEL_NAME)

# Zip and upload the model
model_zip = f'/content/bertopic_model_{MODEL_NAME}.zip'
print(f'Zipping {MODEL_PATH} ...')
with zipfile.ZipFile(model_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(MODEL_PATH):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, os.path.dirname(MODEL_PATH)))
_hf_upload(model_zip, HF_REPO, subfolder=MODEL_NAME)

## Section 10: Keyword Groups

Six thematic groups mirroring `Tables_Civil-War-News.R` — the primary
independent variables in the R regressions.


In [ ]:
KEYWORD_GROUPS = {
    'politics': [
        'elect', 'vote', 'ballot', 'campaign', 'candidate',
        'constitution', 'law', 'legislation', 'congress', 'senate',
        'president', 'governor', 'political', 'federal',
        'amendment', 'rights', 'citizen', 'republican party',
        'democratic party', 'union party',
    ],
    'slavery_direct': [
        'slave', 'bondman', 'bondwoman', 'colored servant', 'unfree',
    ],
    'slavery_ideology': [
        'southern institution', 'peculiar institution', 'southern rights',
        'slave power', 'slave oligarchy', 'slavocracy',
    ],
    'abolition': [
        'abolition', 'emancipation', 'freedman',
        'free negro', 'free black', 'free colored', 'manumission',
    ],
    'race': [
        'negro', 'negress', 'colored man', 'colored woman', 'colored race',
        'mulatto', 'quadroon', 'octoroon', 'miscegenation', 'nigger',
    ],
    'black_military': [
        'negro soldier', 'negro sailor', 'negro troop',
        'black soldier', 'colored soldier', 'colored troop', 'contrabands',
    ],
}
print(f'Keyword groups: {list(KEYWORD_GROUPS.keys())}')

## Section 11: Keyword-Group Cosine Similarity

**Primary measure:** centroid cosine — encode each keyword group as the mean
of its term embeddings, then compute cosine similarity with each article.
Output: one score per group per article, range [−1, 1].

**Stubs (not implemented):** surface keyword counts, KWIC contextual windows,
topic-based cosine. See comments in the original pipeline notebook.


In [ ]:
# embeddings were freed before fit_transform to save RAM; reload from disk.
if 'embeddings' not in dir():
    print(f'Reloading embeddings from {EMBEDDINGS_PATH} ...')
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f'  shape={embeddings.shape}')

print('Encoding keyword group centroids...')
group_centroids = {}
for group, keywords in KEYWORD_GROUPS.items():
    kw_embs = encode_keywords(keywords, embedding_model, CFG)
    group_centroids[group] = kw_embs.mean(axis=0).astype(np.float32)
    print(f'  {group}: {len(keywords)} terms')

# GPU matmul: all groups batched into one (N×D) @ (D×6) → (N×6) op.
# At fp16 the 8.4M×384 array fits in ~6.5 GB VRAM; falls back to
# chunked CPU matmul if torch.cuda is unavailable or OOM.
print('Computing cosine similarities...')
cos_sims         = {}
_centroid_matrix = np.stack(list(group_centroids.values()))  # (n_groups, D)
_group_names     = list(group_centroids.keys())

_gpu_ok = False
if torch.cuda.is_available():
    try:
        _emb_t    = torch.from_numpy(embeddings).cuda().float()  # (N, D) fp32
        _cent_t   = torch.from_numpy(_centroid_matrix).cuda()
        _all_sims = (_emb_t @ _cent_t.T).cpu().numpy()           # (N, n_groups)
        del _emb_t, _cent_t; torch.cuda.empty_cache()
        for j, group in enumerate(_group_names):
            cos_sims[f'cos_sim_{group}'] = _all_sims[:, j]
        del _all_sims
        _gpu_ok = True
        print('  [GPU matmul -- single batched op across all groups]')
    except RuntimeError as _e:
        print(f'  GPU OOM ({_e}), falling back to chunked CPU...')

if not _gpu_ok:
    _CHUNK = 100_000
    for group, centroid in group_centroids.items():
        vals = np.empty(len(embeddings), dtype=np.float32)
        for i2 in range(0, len(embeddings), _CHUNK):
            vals[i2 : i2 + _CHUNK] = embeddings[i2 : i2 + _CHUNK].astype(np.float32) @ centroid
        cos_sims[f'cos_sim_{group}'] = vals

for group in _group_names:
    vals = cos_sims[f'cos_sim_{group}']
    print(f'  cos_sim_{group}: mean={vals.mean():.4f} +/- {vals.std():.4f}')

gc.collect()
print('Cosine similarity done.')

## Section 12: Assemble Results & Export

In [ ]:
try:
    import cudf
    _USE_CUDF = True
    print('cuDF available — using GPU DataFrame ops.')
except ImportError:
    _USE_CUDF = False
    print('cuDF not available — using pandas.')

if _USE_CUDF:
    import pyarrow as pa
    # to_batches() yields proper RecordBatches; from_batches() produces a real pa.Table
    _pa_table  = pa.Table.from_batches(
        news_sub.data.select(['article_id', 'lccn', 'date']).to_batches()
    )
    results_df = cudf.DataFrame.from_arrow(_pa_table)
    del _pa_table
    results_df['topic_id']        = topics
    results_df['embedding_model'] = MODEL_NAME
    # cudf.to_datetime doesn't support errors='coerce' on Series — parse on CPU, push to GPU
    _dates             = pd.to_datetime(results_df['date'].to_pandas(), errors='coerce')
    results_df['date'] = cudf.Series(_dates);  del _dates
    results_df['year_month'] = results_df['date'].dt.strftime('%Y-%m')
    results_df['year']       = results_df['date'].dt.year
else:
    _meta = news_sub.select_columns(['article_id', 'lccn', 'date']).to_pandas()
    results_df = pd.DataFrame({
        'article_id':      _meta['article_id'],
        'lccn':            _meta['lccn'],
        'date':            _meta['date'],
        'topic_id':        topics,
        'embedding_model': MODEL_NAME,
    })
    del _meta
    results_df['date']       = pd.to_datetime(results_df['date'], errors='coerce')
    results_df['year_month'] = results_df['date'].dt.strftime('%Y-%m')
    results_df['year']       = results_df['date'].dt.year.astype('Int64')

if probs is not None and hasattr(probs, '__len__'):
    try:
        p = np.asarray(probs)
        results_df['topic_prob'] = p.max(axis=1) if p.ndim == 2 else p
    except Exception:
        results_df['topic_prob'] = [max(p) if hasattr(p, '__iter__') else float(p) for p in probs]
else:
    results_df['topic_prob'] = float('nan')

for col, vals in cos_sims.items():
    results_df[col] = vals

# Merge sentiment from Notebook 2 (optional)
if SENTIMENT_CSV and os.path.exists(SENTIMENT_CSV):
    sent_df = pd.read_csv(SENTIMENT_CSV)
    merge_cols = ['article_id'] + [c for c in sent_df.columns
                                   if c not in results_df.columns and c != 'article_id']
    if _USE_CUDF:
        results_df = results_df.to_pandas().merge(sent_df[merge_cols], on='article_id', how='left')
        _USE_CUDF = False
    else:
        results_df = results_df.merge(sent_df[merge_cols], on='article_id', how='left')
    print(f'Merged sentiment: {len(merge_cols)-1} columns added')
else:
    print('No sentiment merge (set SENTIMENT_CSV to merge Notebook 2 output).')

# Back to pandas for CSV export
if _USE_CUDF:
    results_df = results_df.to_pandas()

print(f'Results: {len(results_df):,} rows x {len(results_df.columns)} columns')
print('Columns:', list(results_df.columns))

In [ ]:
results_df

In [ ]:
# Exclude topic_id (categorical label — mean is meaningless) and year (constant within year_month)
_no_mean   = {'topic_id', 'year'}
_mean_cols = [c for c in results_df.select_dtypes(include='number').columns
              if c not in _no_mean]

panel_df = (
    results_df
    .groupby(['lccn', 'year_month'])
    .agg(
        n_articles=('article_id', 'count'),
        year=('year', 'first'),
        **{col: (col, 'mean') for col in _mean_cols}
    )
    .reset_index()
)
print(f'Panel: {len(panel_df):,} LCCN-month cells x {len(panel_df.columns)} columns')
print(panel_df.head(3).to_string())

In [ ]:
results_df.to_csv(RESULTS_CSV, index=False)
panel_df.to_csv(PANEL_CSV,    index=False)
print(f'Saved: {RESULTS_CSV}')
print(f'Saved: {PANEL_CSV}')

In [ ]:
# Upload results CSVs under MODEL_NAME subfolder (topic_info already uploaded in Section 9c)
for path in [RESULTS_CSV, PANEL_CSV]:
    if os.path.exists(path):
        _hf_upload(path, HF_REPO, subfolder=MODEL_NAME)
    else:
        print(f'Not found: {path}')

print(f'\nAll uploads complete.')
print(f'Results : hf://datasets/{HF_REPO}/{MODEL_NAME}/')
print(f'Vectors : hf://datasets/{HF_REPO}/{MODEL_NAME}/')